# Loan Default Prediction using Logistic Regression, LDA and QDA
Predictive Analytics course project. **Dataset:** German Credit (UCI, 1000 applicants). **Target:** `default` (1 = bad credit risk, 0 = good).

Stack: Python, pandas, NumPy, SciPy, scikit-learn, statsmodels, Matplotlib, Seaborn.

In [ ]:
import warnings, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from scipy import stats
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             roc_curve, confusion_matrix, ConfusionMatrixDisplay, precision_recall_curve)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
ROOT = Path("..") if Path.cwd().name == "notebooks" else Path(".")
DATA_FILE = ROOT / "data" / "german_credit.csv"

## 1. Data collection
The first run downloads the dataset from UCI (needs `ucimlrepo` and internet) and saves it to `data/german_credit.csv`. Later runs read that file. Raw columns are coded (for example `A11` = checking balance below 0 DM), so they are renamed to readable names.

In [ ]:
NAMES = ["checking_status", "duration", "credit_history", "purpose", "credit_amount", "savings", "employment",
         "installment_rate", "personal_status", "other_debtors", "residence_since", "property", "age",
         "other_plans", "housing", "existing_credits", "job", "num_dependents", "telephone", "foreign_worker"]

if DATA_FILE.exists():
    df = pd.read_csv(DATA_FILE)
else:
    from ucimlrepo import fetch_ucirepo
    repo = fetch_ucirepo(id=144)                      # Statlog (German Credit Data)
    df = repo.data.features.copy()
    df.columns = NAMES
    df["default"] = (repo.data.targets.iloc[:, 0] == 2).astype(int)   # class 2 = bad
    DATA_FILE.parent.mkdir(exist_ok=True)
    df.to_csv(DATA_FILE, index=False)

print(df.shape)
df.info()
print(df["default"].value_counts(normalize=True).round(3))
df.head()

**Data challenges:** the classes are imbalanced (about 70% repaid), amounts are in Deutsche Mark, and many fields are coded categories, so they need encoding before modeling.

## 2. Data preprocessing: missing values and outliers (IQR, box plot)

In [ ]:
num_cols = ["duration", "credit_amount", "age", "installment_rate", "residence_since", "existing_credits", "num_dependents"]
cat_cols = [c for c in df.columns if c not in num_cols + ["default"]]

print("Missing values:", int(df.isna().sum().sum()))
for c in num_cols:
    df[c] = df[c].fillna(df[c].median())          # median for numeric columns
for c in cat_cols:
    df[c] = df[c].fillna(df[c].mode()[0])         # mode for categorical columns

In [ ]:
plot_cols = ["duration", "credit_amount", "age"]
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for a, c in zip(ax, plot_cols):
    sns.boxplot(y=df[c], ax=a, color="#9ab6ff"); a.set_title(c)
plt.suptitle("Box plots before capping"); plt.show()

for c in plot_cols:
    q1, q3 = df[c].quantile([.25, .75]); iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    print(f"{c}: {int(((df[c] < lo) | (df[c] > hi)).sum())} outliers, IQR fences ({lo:.1f}, {hi:.1f})")
    df[c] = df[c].clip(lo, hi)                    # cap at the fences (winsorize)

## 3. Exploratory data analysis: mean, median, mode, variance, standard deviation, z-score

In [ ]:
summary = pd.DataFrame({"mean": df[num_cols].mean(), "median": df[num_cols].median(),
                        "mode": df[num_cols].mode().iloc[0], "variance": df[num_cols].var(),
                        "std": df[num_cols].std(), "skew": df[num_cols].skew()}).round(2)
display(summary)
z = np.abs(stats.zscore(df[plot_cols]))
print("Rows with any |z| > 3 after capping:", int((z > 3).any(axis=1).sum()))

## 4. Data visualization: bar chart, histogram, pie chart, heatmap, scatter plot

In [ ]:
labels = {0: "Repaid", 1: "Default"}
colors = ["#5ec8c0", "#ff7a6b"]
fig, ax = plt.subplots(2, 3, figsize=(16, 9))
df["default"].value_counts().rename(labels).plot.bar(ax=ax[0, 0], color=colors, title="Bar chart: outcomes")
df["credit_amount"].plot.hist(bins=25, ax=ax[0, 1], color="#6f8fe8", title="Histogram: credit amount")
df["default"].value_counts().rename(labels).plot.pie(ax=ax[0, 2], autopct="%1.1f%%", colors=colors, title="Pie chart: class share")
sns.heatmap(df[num_cols + ["default"]].corr(), annot=True, fmt=".2f", cmap="coolwarm", ax=ax[1, 0]); ax[1, 0].set_title("Heatmap: correlations")
sns.scatterplot(data=df, x="duration", y="credit_amount", hue="default", palette=colors, ax=ax[1, 1]); ax[1, 1].set_title("Scatter: duration vs amount")
pd.crosstab(df["checking_status"], df["default"], normalize="index").plot.bar(stacked=True, ax=ax[1, 2], color=colors, title="Default rate by checking account")
plt.tight_layout(); plt.show()

## 5. Feature engineering
New features: monthly repayment burden, age groups, employment length in years (from the coded bands), and a long-loan flag.

In [ ]:
EMP_YEARS = {"A71": 0, "A72": 0.5, "A73": 2.5, "A74": 5.5, "A75": 9}
df["monthly_burden"] = df["credit_amount"] / df["duration"]
df["age_group"] = pd.cut(df["age"], [0, 25, 35, 50, 120], labels=["under26", "26-35", "36-50", "over50"]).astype(str)
df["employment_years"] = df["employment"].map(EMP_YEARS)
df["long_loan"] = (df["duration"] > 24).astype(int)

X = pd.get_dummies(df.drop(columns=["default"]), drop_first=True).astype(float)
y = df["default"]
print("Features after encoding:", X.shape[1])

## 6. Train/test split and feature selection
The split comes first and selection is fitted on the training data only, so the test set never influences which features are chosen.

In [ ]:
X_tr_all, X_te_all, y_tr, y_te = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

selector = SelectKBest(f_classif, k=15).fit(X_tr_all, y_tr)
scores = pd.Series(selector.scores_, index=X.columns).sort_values(ascending=False)
scores.head(15).plot.barh(figsize=(7, 5), title="Top 15 features (ANOVA F-score)").invert_yaxis(); plt.show()

selected = list(X.columns[selector.get_support()])
scaler = StandardScaler().fit(X_tr_all[selected])
X_tr = pd.DataFrame(scaler.transform(X_tr_all[selected]), columns=selected, index=X_tr_all.index)
X_te = pd.DataFrame(scaler.transform(X_te_all[selected]), columns=selected, index=X_te_all.index)
selected

## 7. Logistic regression: coefficient estimation (maximum likelihood) and odds ratios

In [ ]:
logit = sm.Logit(y_tr, sm.add_constant(X_tr)).fit(disp=0)
print(logit.summary())
odds = pd.DataFrame({"coef": logit.params, "odds_ratio": np.exp(logit.params), "p_value": logit.pvalues}).round(3)
odds.sort_values("odds_ratio", ascending=False)

**How to read it:** an odds ratio above 1 means that a one standard deviation increase in the feature raises the odds of default. Below 1 lowers them. Small p-values (under 0.05) mark features whose effect is statistically significant.

## 8. Model comparison: Logistic Regression vs LDA vs QDA

In [ ]:
models = {"Logistic Regression": LogisticRegression(max_iter=1000),
          "LDA": LinearDiscriminantAnalysis(),
          "QDA": QuadraticDiscriminantAnalysis(reg_param=0.1)}   # regularised: QDA needs a covariance matrix per class
rows, probs = [], {}
for name, m in models.items():
    m.fit(X_tr, y_tr)
    pred, prob = m.predict(X_te), m.predict_proba(X_te)[:, 1]
    probs[name] = prob
    rows.append({"Model": name, "Accuracy": accuracy_score(y_te, pred), "Precision": precision_score(y_te, pred),
                 "Recall": recall_score(y_te, pred), "F1": f1_score(y_te, pred), "ROC-AUC": roc_auc_score(y_te, prob)})
results = pd.DataFrame(rows).set_index("Model").round(3)
results

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(20, 4.5))
for name, prob in probs.items():
    fpr, tpr, _ = roc_curve(y_te, prob)
    ax[0].plot(fpr, tpr, label=f"{name} (AUC {roc_auc_score(y_te, prob):.2f})")
ax[0].plot([0, 1], [0, 1], "k--"); ax[0].legend(); ax[0].set_title("ROC curves")
for a, (name, m) in zip(ax[1:], models.items()):
    ConfusionMatrixDisplay(confusion_matrix(y_te, m.predict(X_te)), display_labels=["Repaid", "Default"]).plot(ax=a, colorbar=False)
    a.set_title(name)
plt.tight_layout(); plt.show()

cv = {n: cross_val_score(make_pipeline(StandardScaler(), m), X[selected], y, cv=5, scoring="roc_auc").mean().round(3)
      for n, m in models.items()}
print("5-fold cross-validated ROC-AUC:", cv)

**Why LDA and QDA differ:** LDA assumes both classes share one covariance matrix (a linear boundary). QDA fits a separate covariance matrix per class (a curved boundary), which needs more data and can overfit on 800 training rows.

## 9. Choosing the approval cut-off
A lender can trade missed defaults against rejected good customers by moving the probability cut-off.

In [ ]:
p, r, t = precision_recall_curve(y_te, probs["Logistic Regression"])
plt.plot(t, p[:-1], label="Precision"); plt.plot(t, r[:-1], label="Recall")
plt.xlabel("Default-probability cut-off"); plt.legend(); plt.title("Precision and recall vs cut-off"); plt.show()

## 10. Ethical considerations: bias check

In [ ]:
audit = df.loc[X_te.index, ["age_group"]].copy()
audit["actual"] = y_te
audit["predicted_default"] = models["Logistic Regression"].predict(X_te)
display(audit.groupby("age_group").agg(applicants=("actual", "size"), actual_default_rate=("actual", "mean"),
                                      predicted_default_rate=("predicted_default", "mean")).round(3))

no_age = [c for c in selected if c != "age" and not c.startswith("age_group")]
fair_model = LogisticRegression(max_iter=1000).fit(X_tr[no_age], y_tr)
print("ROC-AUC with age features:   ", round(roc_auc_score(y_te, probs["Logistic Regression"]), 3))
print("ROC-AUC without age features:", round(roc_auc_score(y_te, fair_model.predict_proba(X_te[no_age])[:, 1]), 3))

Compare the predicted default rate across age groups. If dropping age costs little accuracy, a lender has a strong reason to leave it out. Also consider foreign-worker status and personal status (which encodes sex), which raise the same fairness concerns.

## 11. Export coefficients for the web app
This fits a simpler logistic regression on the seven inputs the web app asks for and writes `model.json`, which `index.html` loads. Employment bands are converted to approximate years, and amounts stay in Deutsche Mark.

In [ ]:
CHECK = {"A11": "neg", "A12": "low", "A13": "high", "A14": "none"}
SAVE = {"A61": "low", "A62": "mid", "A63": "mid", "A64": "high", "A65": "mid"}

web = pd.DataFrame({"amount": df["credit_amount"], "duration": df["duration"], "age": df["age"],
                    "rate": df["installment_rate"], "job": df["employment_years"]})
Z = (web - web.mean()) / web.std()
Z["checking"] = df["checking_status"].map(CHECK)
Z["savings"] = df["savings"].map(SAVE)
D = pd.get_dummies(Z, columns=["checking", "savings"]).astype(float).drop(columns=["checking_low", "savings_mid"])
fit = sm.Logit(y, sm.add_constant(D)).fit(disp=0)
b = fit.params

model = {"intercept": float(b["const"]),
         "num": {c: {"mean": float(web[c].mean()), "sd": float(web[c].std()), "b": float(b[c])} for c in web},
         "checking": {"neg": float(b["checking_neg"]), "low": 0.0, "high": float(b["checking_high"]), "none": float(b["checking_none"])},
         "savings": {"low": float(b["savings_low"]), "mid": 0.0, "high": float(b["savings_high"])}}
(ROOT / "model.json").write_text(json.dumps(model, indent=2))
print(json.dumps(model, indent=2))

## Conclusion
Write 4-5 lines here: which model performed best and why, the strongest risk factors from the odds ratios, what the cut-off analysis shows, and what the bias check tells you about using age in a lending model.